# ABM synthetic data → Bayesian event-rate inference

<!-- Generate independent ABM realizations, keep one count per killer cell, and fit the population distribution of effective event rates. Choose `contacts`, `synapses`, or `kills` below. Rates are per minute when using the ABM's baseline units.

This first framework fits the four existing Orca count models and an optional single-population two-Gamma mixture. It estimates descriptive event-rate parameters; ABM damage, binding, and exhaustion parameters have different meanings.

From the repository root, install dependencies if needed: `python -m pip install -e '.[all]' jupyter`. Open this notebook using that Python environment and run all cells. -->

In [ ]:
from pathlib import Path
from dataclasses import replace
from tempfile import mkdtemp
import json
import os
import sys
os.environ["PYTENSOR_FLAGS"] = (
    "cxx=/Users/marsian/PhD_LifeSciRes/MantiShrimp/paper/section_3/inference/pytensor-clang++,"
    "mode=FAST_RUN,linker=cvm"
)

import pytensor


ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "src/mantishrimp").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the MantiShrimp checkout.")
SECTION = ROOT / "paper/section_3"
sys.path[:0] = [str(SECTION), str(ROOT / "src")]

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
from IPython.display import display
from tqdm.auto import tqdm

from mantishrimp import *
from inference import inference as cih

import scienceplots
plt.style.use(["science"])

OUT = Path("outputs/test")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
NUM_OBS_REQUIRED = 500

config = SimulationConfig(
    n_killers=50,
    n_targets=200,
    killer_radius=12.5,
    target_radius=15.0,
    duration=10.0,
    max_dt=1 / 25,
    max_drift_displacement=1.0,
    seed=None,
    initial_separation_factor=4,
    domain=DomainConfig(
        x_min=-1200, x_max=1200,
        y_min=-1200, y_max=1200,
        boundary="confined",
    ),
    motility=MotilityConfig(
        killer_polarity_decay=2.5, target_polarity_decay=5.0,
        killer_polarity_noise=2.0, target_polarity_noise=0.75,
        killer_speed=175.0, target_speed=125.0,
        killer_translation_noise=0.75, target_translation_noise=0.2,
    ),
    mechanics=MechanicsConfig(
        binding_rate=5.0,    # k_on: synapse formation while cells are close enough
        unbinding_rate=0.5,  # k_off: synapse dissociation
    ),
    cytotoxicity=CytotoxicityConfig(
        instant_kill=False,          # Accumulate damage before killing.
        # Killer activity
        initial_killer_state=1.0,   # All killers start fully active.
        killer_state_sd=0.0,        # Increase for initial cell-to-cell variation.
        kill_probability=1.0,      # Every new conjugation is productive.
        # Damage production
        killing_rate_min=0.0,      # Damage rate at killer activity = 0.
        killing_rate_max=5.0,      # Damage rate at killer activitya = 1.
        # Exhaustion during productive conjugation
        exhaustion_rate_min=0.01, # Rate when target state = 0.
        exhaustion_rate_max=0.1,   # Rate when target state = 1.
        # Target susceptibility, damage, and repair
        initial_target_state=1.0,
        initial_death_factor=0.0,  # Start undamaged.
        death_threshold=1.0,      # Die when accumulated damage reaches 1.
        recovery_rate=0,        # Example repair rate; 0 disables repair.
    ),
)

n_runs = NUM_OBS_REQUIRED // config.n_killers + 1
seeds = np.random.SeedSequence(config.seed).spawn(n_runs)
tables = []

for run_id, child_seed in tqdm(enumerate(seeds, start=1)):
    run_seed = int(child_seed.generate_state(1)[0])
    run_config = replace(config, seed=run_seed)

    # print(f"\nSimulation {run_id}/{n_runs}")
    result = simulate(run_config, progressbar=True)
    kills = result.kills_per_cell()  # Includes zero-count killers.

    tables.append(pd.DataFrame({
        "simulation": run_id,
        "seed": run_seed,
        "killer_id": np.arange(len(kills)),
        "kills": kills,
        "exposure": result.duration,
    }))

kills_data = pd.concat(tables, ignore_index=True)
kills_data.to_csv(OUT / "kills_data.csv", index=False)
assert len(kills_data) > NUM_OBS_REQUIRED

counts = kills_data["kills"].to_numpy()
exposure = config.duration  # Same observation duration for every run.

print(f"\nCollected {len(counts)} killer-cell observations.")
display(kills_data.head())
display(
    kills_data.groupby("simulation")["kills"].agg(
        n_killers="size",
        total_kills="sum",
        mean_kills="mean",
    )
)

In [ ]:
from mantishrimp.animation import animate

plt.rcParams["animation.embed_limit"] = 100  # Maximum embedded HTML size, MB.

animation_params = {
    # Displayed cell sizes
    "killer_radius": result.config.killer_radius,
    "target_radius": result.config.target_radius,
    "dead_target_radius": result.config.target_radius,

    # Colours and outlines
    "killer_colour": "#92C4E9",
    "live_target_colour": "#96C36E",
    "dead_target_colour": "#EA9A9D",
    "cir_linewidth": 0.8,

    # Trajectory trails
    "traj_steps": 250,          # Recorded snapshots retained in each trail.
    "traj_linewidth": 2.0,
    "traj_alpha": 0.45,         # Transparency: 0=invisible, 1=opaque.

    # Figure and margins
    "figsize": (6, 7),          # Width, height in inches.
    "fig_dpi": 100,
    "plot_margin": 0.06,        # Padding as a fraction of domain size.
    # "ax_box": (-1500, 1500, -1500, 1500),  # Optional exact axis limits.

    # Playback
    "interval": 40,             # Milliseconds between displayed frames.
    "frame_step": 10,           # Render every tenth recorded snapshot.

    # Legend
    "add_swatch_legend": True,
    "legend_labels": ["Killer", "Live target", "Dead target"],

    # Display and export
    "render_html": True,
    "save_gif_path": None,      # Or "outputs/simulation.gif".
    "gif_fps": 25,              # Saved GIF playback speed.
    "transparent_background": True
}

animation = animate(result, **animation_params)

In [ ]:
frequencies = np.bincount(counts) / len(counts)
x = np.arange(len(frequencies))

fig, ax = plt.subplots(figsize=(8, 3), dpi=350)
ax.bar(
    x, frequencies, width=0.8,
    color="#85a993", edgecolor="black", linewidth=0.8,
)
ax.set_xticks(x)
ax.set_xlim(-0.6, x[-1] + 0.6)
ax.set_xlabel("Kills per NK cell", fontsize=28)
ax.set_ylabel("Proportion", fontsize=28)
ax.tick_params(axis="both", labelsize=22)
ax.minorticks_off()

fig.tight_layout()
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
plt.show()

In [ ]:
kills_data = pd.read_csv(OUT / "kills_data.csv")
counts = kills_data["kills"].to_numpy()
times = kills_data["exposure"].to_numpy()
OBS_TIME = float(times[0])
assert np.isfinite(times).all() and OBS_TIME > 0
assert np.all(times == OBS_TIME), "This module requires equal exposure times."

MODEL_ORDER = ["homo", "Z2P", "Dis2P", "hetero3"]
functions = {
    "homo": cih.inference_homo,
    "Z2P": cih.inference_Z2P,
    "Dis2P": cih.inference_Dis2P,
    "hetero3": cih.inference_hetero3,
}
sampling = dict(
    draws=5_000,
    chains=6,
    cores=6,
    lambda_prior_bounds=(-5.0, 2.0),
    random_seed=None,
    threshold=0.5,
    correlation_threshold=0.01,
)

posterior_data, rows = {}, []

for name in MODEL_ORDER:
    kwargs = sampling.copy()
    if name in {"Z2P", "hetero3"}:
        kwargs["p_prior_bounds"] = (1.0, 1.0)
    if name in {"Dis2P", "hetero3"}:
        kwargs["std_prior_factor"] = 3.0

    print(f"Running {name}")
    fn = getattr(cih, f"inference_{name}")
    idata = functions[name](counts, OBS_TIME, **kwargs)["idata"]

    posterior_data[name] = idata
    idata.to_netcdf(OUT / f"posterior_{name}.nc")
    az.summary(idata).to_csv(OUT / f"summary_{name}.csv")
    rows.append((name, cih.smc_log_evidence(idata)))


In [ ]:
logml_table = pd.DataFrame(rows, columns=["model", "logml"])
logml_table = logml_table.sort_values("logml", ascending=False)
display(logml_table)

plot_df = logml_table.sort_values("logml")
values = (plot_df["logml"] - logml_table["logml"].max()) / np.log(10)
best_model = logml_table.iloc[0]["model"]

colors = {
    "homo": "#85a993", "Z2P": "#f7c435",
    "Dis2P": "#c75a1b", "hetero3": "#92C4E9",
}
xmin = -9
bf3 = np.log10(3)

fig, ax = plt.subplots(figsize=(9, 7), dpi=400)
ax.set_xlim(xmin, 0.35)

for left, right, color, alpha in [
    (-1, -bf3, "#FFF3B0", 0.34),
    (-2, -1, "#FFD28A", 0.30),
    (xmin, -2, "#E76F51", 0.26),
]:
    ax.axvspan(left, right, color=color, alpha=alpha, linewidth=0, zorder=0)

for threshold in [-bf3, -1, -2]:
    ax.axvline(threshold, color="black", linestyle=":", linewidth=1.2, alpha=0.65)
ax.axvline(0, color="black", linewidth=1.8, zorder=3)

ax.barh(
    plot_df["model"], np.maximum(values, xmin),
    color=["black" if m == best_model else colors[m] for m in plot_df["model"]],
    edgecolor="black", linewidth=1.5, alpha=0.9, zorder=2,
)

for y, value in enumerate(values):
    clipped = value < xmin
    x = xmin + 0.18 if clipped else (value + 0.12 if value < -0.25 else 0.05)
    label = f"// {value:.2f}" if clipped else f"{value:.2f}"
    ax.text(x, y, label, va="center", ha="left", fontsize=26, zorder=5)

ax.set_xlabel(
    r"$\log_{10}\mathrm{BF}(\mathcal{M}/\mathcal{M}_{\mathrm{best}})$",
    fontsize=30,
)
ax.tick_params(axis="x", labelsize=22)
ax.tick_params(axis="y", labelsize=28)
ax.minorticks_off()
ax.grid(False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.8)
    spine.set_color("black")

fig.subplots_adjust(left=0.22, bottom=0.20)
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
plt.show()